### Specify Inputs

In [274]:
# This is the experiment code we want to look at.
search = "MLB000|MLB003|QSC022|MLB|MLD023|MLD007|FCD|MLD012|MLD000|MLD025|MLD026|MLD016|MLD018" #for all 6L baseline cells
# search = "MLD012|MLD016|MLD018|FCD00[0-5]|FCD009|MLB007" #for 22L cells
# search = "MLD023" #for 6L baseline builds with Cathy cells


GroupBy = 'Experiment'  #Group cells by Experiment code (MLB006, MLD023, FCD00, etc) in screen and cycle plots
#GroupBy = 'WorkWeek'  #Group cells by work week built in screen and cycle plots
#GroupBy = 'Batch'   #Group cells by batches (MLB006AA, MLB006AB, MLB00AC, etc) in screen and cycle plots


CellsToExclude = ['MLD023AB-PS00-01', 'MLD023AB-PS00-02', 'MLD023AB-PS00-05', 'QSC022AG-PS00-03', 'MLB006AP-PS00-04'] #Cells that should not be included in BOTH yield and reliability cycling analyses and plots
CellsThatPassedScreen = ['MLD007AW-PS00-02', 'MLD007AX-PS00-01', 'MLD007AX-PS00-02', 'MLD007BA-PS00-01', 'MLD007BA-PS00-02', 'MLD007BA-PS00-04', 'MLB006AH-PS00-03',  'MLD023AC-PS00-02','MLD023AD-PS00-01','MLD023AD-PS00-02','MLD023AD-PS00-03', 'MLD023AD-PS00-04', 'MLD023AD-PS00-05'] # List of cells that did not fail screen despite datahub saying it did
CellsThatFailedScreen = ['MLD023AF-PS00-03', 'MLD023AB-PS00-03'] # List of cells that should have failed screen despite datahub saying it didn't (i.e. "soft-shorted" during screening)

builtby = "7/01/2024" #search for cells built on or after this date


### Import Functions and Modules

In [275]:
#Import libraries and functions
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.utils import restricted_mean_survival_time
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import plotly.io as pio
import plotly.graph_objects as go
from qsdc.client import Client
from datetime import datetime
import warnings


pio.templates.default = "plotly_white"
clrs = plotly.colors.DEFAULT_PLOTLY_COLORS
qs_client = Client()

#Remove warning messages
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
# Convert builtby to datetime
builtby_date = pd.to_datetime(builtby, format="%m/%d/%Y")

### Pull from Datahub


In [276]:
#Pull multilayer screen data
MLScreenEach = qs_client.data_hub.get_dataset(dataset = 'MFG-80L-ML-SCREEN-CYCLE') ##electrical test data of unit cells for each cycle
MLScreenSummary = qs_client.data_hub.get_dataset(dataset = 'MFG-80L-ML-SCREEN') ##electrical test data of unit cells that sumamrizes each screen test
#Pull multilayer track cycle data
MLTrackCycleEach = qs_client.data_hub.get_dataset(dataset = 'MFG-80L-TRACK-CYCLE-REL-CYCLE')
MLTrackCycleSummary = qs_client.data_hub.get_dataset(dataset = 'MFG-80L-TRACK-CYCLE-REL')
#Pull multilayer 1C cycle data
MLslctCycleSummary = qs_client.data_hub.get_dataset(dataset = 'MFG-80L-ML-SLCT')
MLslctCycleEach = qs_client.data_hub.get_dataset(dataset = 'MFG-80L-ML-SLCT-CYCLE')

#Pull geneology/multilayer info of unit cells
dmlg = qs_client.data_hub.get_dataset(dataset = 'MFG-80L-ML-PRODUCTION') ##multilayer info (ML_id)
dmlg = dmlg.drop_duplicates(subset='US_id')


### Create Master Spreadsheets

In [277]:
##Create a Master Spreadsheet for Screening data
# Grab electrical metrics by consolidating individual cycles in screen
def last_non_missing(series):
    return series.dropna().iloc[-1] if not series.dropna().empty else None
def minimum_non_missing(series):
    return series.dropna().min() if not series.dropna().empty else None
# Group by cell id and consolidate
dfc_consolidated = (
    MLScreenEach.groupby('MLT_id', as_index=False)
    .agg({
        'AMSDcCapacity': last_non_missing,
        'DischargeCapacity': last_non_missing,
        'dvdt':minimum_non_missing, 
        'MedDcASR': last_non_missing
    })
)
dfc_consolidated = dfc_consolidated.rename(columns={
    'AMSDcCapacity': 'CellSpecificDischargeCapacity',
    'DischargeCapacity': 'CellCo3DischargeCapcity',
    'dvdt': 'Celldvdt',
    'MedDcASR': 'Cell1CDischargeASR'
})
MLScreenSummary = MLScreenSummary.merge(
    dfc_consolidated,
    left_on='MLT_id',
    right_on='MLT_id',
    how='inner'  # Use 'inner' to keep only matching rows; adjust to 'left' or 'outer' as needed
)

#Filter by what we are searching for
filtered_MLScreenSummary = MLScreenSummary[MLScreenSummary['MLT_id'].str.contains(search, regex=True)]
filtered_MLScreenEach = MLScreenEach[MLScreenEach['MLT_id'].str.contains(search, regex=True)]
#Overwrite screen yield for cells that didn't really short
columns_to_overwrite = ['stage1_yield', 'stage2_yield', 'stage3_yield', 'stage4_yield', 'stage5_yield'] # Specify columns to overwrite
filtered_MLScreenSummary.loc[filtered_MLScreenSummary['MLT_id'].isin(CellsThatPassedScreen), columns_to_overwrite] = 1 # Overwrite the specified columns with 1 for matching rows
filtered_MLScreenSummary.loc[filtered_MLScreenSummary['MLT_id'].isin(CellsThatPassedScreen), ['CycleFailure', 'AnyFailure']] = False # Overwrite the specified columns with 1 for matching rows
# Suppose CellsThatFailedScreen is a list of strings
filtered_MLScreenSummary.loc[filtered_MLScreenSummary['MLT_id'].isin(CellsThatFailedScreen), columns_to_overwrite] = 0 # Overwrite the specified columns with 1 for matching rows
filtered_MLScreenSummary.loc[filtered_MLScreenSummary['MLT_id'].isin(CellsThatFailedScreen), ['CycleFailure', 'AnyFailure']] = True # Overwrite the specified columns with 1 for matching rows

columns_to_overwrite2 = ['AnyFailure', 'CycleFailure'] # Specify columns to overwrite
filtered_MLScreenSummary.loc[filtered_MLScreenSummary['MLT_id'].isin(CellsThatPassedScreen), columns_to_overwrite2] = False

# Exclude cells
MLScreenMaster = filtered_MLScreenSummary[~filtered_MLScreenSummary['MLT_id'].isin(CellsToExclude)] 

# Add additional columns that are typically used for grouping
MLScreenMaster['Experiment']= MLScreenMaster['MLT_id'].str[:6]  #Group by experiment
MLScreenMaster['Batch']= MLScreenMaster['MLT_id'].str[:8]  #Group by experiment
MLScreenMaster['WorkWeek']= MLScreenMaster['TestCycleStart_week_first']  #Group by experiment


# Group screen yield by:
MLScreenMaster['Group']= MLScreenMaster[GroupBy]  #Group by experiment



In [278]:
##Create a Master Spreadsheets for Cycling Data

#Filter by what we are searching for
MLTrackCycleSummary = MLTrackCycleSummary[MLTrackCycleSummary['sample_id'].str.contains(search, regex=True)]
MLTrackCycleEach = MLTrackCycleEach[MLTrackCycleEach['sample_id'].str.contains(search, regex=True)]
MLslctCycleSummary = MLslctCycleSummary[MLslctCycleSummary['MLT_id'].str.contains(search, regex=True)]
MLslctCycleEach = MLslctCycleEach[MLslctCycleEach['MLT_id'].str.contains(search, regex=True)]

# Exclude cells
MLTrackCycleSummary = MLTrackCycleSummary[~MLTrackCycleSummary['sample_id'].isin(CellsToExclude)] 
MLTrackCycleEach = MLTrackCycleEach[~MLTrackCycleEach['sample_id'].isin(CellsToExclude)] 
MLslctCycleSummary = MLslctCycleSummary[~MLslctCycleSummary['MLT_id'].isin(CellsToExclude)] 
MLslctCycleEach = MLslctCycleEach[~MLslctCycleEach['MLT_id'].isin(CellsToExclude)] 


non_empty_failures = [cell for cell in CellsThatFailedScreen if cell.strip()]
if non_empty_failures:
    pattern = '|'.join(non_empty_failures)
    MLTrackCycleSummary = MLTrackCycleSummary[~MLTrackCycleSummary['sample_id'].str.contains(pattern)]
    MLTrackCycleEach = MLTrackCycleEach[~MLTrackCycleEach['sample_id'].str.contains(pattern)]
    MLslctCycleSummary = MLslctCycleSummary[~MLslctCycleSummary['MLT_id'].str.contains(pattern)]
    MLslctCycleEach = MLslctCycleEach[~MLslctCycleEach['MLT_id'].str.contains(pattern)]


MLTrackCycleEach.loc[MLTrackCycleEach['MiscTestAnomaly'] == 1, ['dvdt_failure', 'min_track_cycle_voltage_failure', 'CapacityChargeFraction_failure'] ] = [0, 0, 0]

# Add additional columns that are typically used for grouping
MLTrackCycleEach['Experiment']= MLTrackCycleEach['sample_id'].str[:6]  #Group by experiment
MLTrackCycleEach['Batch']= MLTrackCycleEach['sample_id'].str[:8]  #Group by experiment
MLTrackCycleEach['WorkWeek']= MLTrackCycleEach['TestCycleStart_week_first']  #Group by experiment
MLslctCycleEach['Experiment']= MLslctCycleEach['MLT_id'].str[:6]  #Group by experiment
MLslctCycleEach['Batch']= MLslctCycleEach['MLT_id'].str[:8]  #Group by experiment
MLslctCycleEach['WorkWeek']= MLslctCycleEach['TestCycleStart_week_first']  #Group by experiment

MLTrackCycleSummary['Experiment']= MLTrackCycleSummary['sample_id'].str[:6]  #Group by experiment
MLTrackCycleSummary['Batch']= MLTrackCycleSummary['sample_id'].str[:8]  #Group by experiment
MLTrackCycleSummary['WorkWeek']= MLTrackCycleSummary['TestCycleStart_week_first']  #Group by experiment
MLslctCycleSummary['Experiment']= MLslctCycleSummary['MLT_id'].str[:6]  #Group by experiment
MLslctCycleSummary['Batch']= MLslctCycleSummary['MLT_id'].str[:8]  #Group by experiment
MLslctCycleSummary['WorkWeek']= MLslctCycleSummary['TestCycleStart_week_first']  #Group by experiment

# Group track cycles by:
MLTrackCycleEach['Group']= MLTrackCycleEach[GroupBy]  #Group by experiment
MLslctCycleEach['Group']= MLslctCycleEach[GroupBy]  #Group by experiment
MLTrackCycleSummary['Group']= MLTrackCycleSummary[GroupBy]  #Group by experiment
MLslctCycleSummary['Group']= MLslctCycleSummary[GroupBy]  #Group by experiment


In [279]:
##Tier Each ML Pouch
#Pull geneology/multilayer info
dmlg = dmlg.drop_duplicates(subset='US_id')
CellsInML = MLScreenMaster.merge(dmlg, left_on='MLT_id', right_on='ML_id', how='left')
CellsInML = CellsInML[['MLT_id', 'US_id']].rename(columns={'US_id': 'Cell ID'})

#Pull cell metrology data from datahub, both standard/auto metrology and manual review
dfctq = qs_client.data_hub.get_dataset(dataset = 'MFG-60L-UC-CTQ') ## standard metro

dfctq_filtered = dfctq[dfctq['US_id'].isin(CellsInML['Cell ID'])]

yielded_dfctq = dfctq_filtered[dfctq_filtered['unit_cell_test_yield'] == 1] #keep cells that yielded

dfmr = qs_client.data_hub.get_dataset(dataset = 'MFG-60L-UC-MR') ## manual review
dfmr_filtered = dfmr[dfmr['US_id'].isin(CellsInML['Cell ID'])]


yielded_dfmr = dfmr_filtered[dfmr_filtered['unit_cell_test_yield'] == 1] #keep cells that yielded


# First, merge the DataFrames on 'US_id' to align rows
merged_df = yielded_dfctq.merge(yielded_dfmr[['US_id', 'edge_thickness_tier_us_mr', 'A1_anode_tier_top_us_mr', 'A1_anode_tier_bottom_us_mr',
                                              'cathode_alignment_custom_model_tier_us_mr', 'median_contour_catholyte_pct_us_mr', 'disposition_mr', 'failure_modes_mr']], on='US_id', how='left')
# Then, overwrite 'edge_thickness_tier_us' in 'filtered_dfctq' where 'edge_thickness_tier_us_mr' has a value
merged_df['edge_thickness_tier_us'] = merged_df['edge_thickness_tier_us_mr'].combine_first(merged_df['edge_thickness_tier_us'])
# Then, overwrite 'A1_anode_tier_top_us' in 'filtered_dfctq' where 'A1_anode_tier_top_us_mr' has a value
merged_df['A1_anode_tier_top_us'] = merged_df['A1_anode_tier_top_us_mr'].combine_first(merged_df['A1_anode_tier_top_us'])
# Then, overwrite 'A1_anode_tier_bottom_us' in 'filtered_dfctq' where 'A1_anode_tier_bottom_us_mr' has a value
merged_df['A1_anode_tier_bottom_us'] = merged_df['A1_anode_tier_bottom_us_mr'].combine_first(merged_df['A1_anode_tier_bottom_us'])
# Then, overwrite 'A1_anode_tier_bottom_us' in 'filtered_dfctq' where 'A1_anode_tier_bottom_us_mr' has a value
merged_df['cathode_alignment_custom_model_tier_us'] = merged_df['cathode_alignment_custom_model_tier_us_mr'].combine_first(merged_df['cathode_alignment_custom_model_tier_us'])
# Then, overwrite 'A1_anode_tier_bottom_us' in 'filtered_dfctq' where 'A1_anode_tier_bottom_us_mr' has a value
merged_df['median_contour_catholyte_pct_us'] = merged_df['median_contour_catholyte_pct_us_mr'].combine_first(merged_df['median_contour_catholyte_pct_us'])
# Then, overwrite 'disposition_us' in 'disposition_mr' has a value
merged_df['disposition'] = merged_df['disposition_mr'].combine_first(merged_df['disposition'])
# Then, overwrite 'disposition_us' in 'disposition_mr' has a value
merged_df['failure_modes'] = merged_df['failure_modes_mr'].combine_first(merged_df['failure_modes'])
# Drop the 'edge_thickness_tier_us_mr' column if you don't need it
dfctq_updated = merged_df.drop(columns=['edge_thickness_tier_us_mr', 'A1_anode_tier_top_us_mr', 'A1_anode_tier_bottom_us_mr',
                                              'cathode_alignment_custom_model_tier_us_mr',  'disposition_mr','failure_modes_mr' ])

#Update Final Tier of Cells
conditions = [
    dfctq_updated['disposition'] == 'Tier 1',
    dfctq_updated['disposition'] == 'Tier 2',
    dfctq_updated['disposition'] == 'Fail',
    dfctq_updated['disposition'] == 'Scrap',
    dfctq_updated['disposition'] == 'Missing Data',
]
choices = ['1', '2', '3','Scrapped', 'TBD']
dfctq_updated['Tier'] = np.select(conditions, choices)



# Merge dfctq_updated['US_id', 'Tier'] with CellsInML based on 'US_id'
CellsInML = CellsInML.merge(dfctq_updated[['US_id', 'Tier', 'center_normalized_0_5mm_eroded_rect_outside_median_us', 'median_contour_catholyte_pct_us_mr']], 
                            left_on='Cell ID', 
                            right_on='US_id', 
                            how='left')


# Assign the 'Tier' column to 'Cell Tier' and drop the extra 'US_id' column
CellsInML['Cell Tier'] = CellsInML['Tier']
CellsInML = CellsInML.drop(columns=['Tier', 'US_id'])

# Group by "samplename" and find the max "Cell Tier" for each
CellsInML['Cell Tier'] = pd.to_numeric(CellsInML['Cell Tier'], errors='coerce')
FinalMLTier = CellsInML.groupby('MLT_id', as_index=False)['Cell Tier'].max()
FinalMLThickness = CellsInML.groupby('MLT_id', as_index=False)['center_normalized_0_5mm_eroded_rect_outside_median_us'].max()

# Rename columns as required
FinalMLTier.columns = ['Multilayer', 'ML Tier']
# Convert "ML Tier" to integer and format as "Tier {max Cell Tier}"
# Conditionally update 'ML Tier'
FinalMLTier['ML Tier'] = np.where(
    FinalMLTier['ML Tier'].isna(), 
    np.nan,  # Keep NaN if it was originally NaN
    "Tier " + FinalMLTier['ML Tier'].fillna(0).astype(int).astype(str)
)

# Merge df_master with FinalMLTier on "samplename" and "Multilayer"
MLScreenMaster= MLScreenMaster.merge(FinalMLTier, left_on='MLT_id', right_on='Multilayer', how='left')
# Merge df_master with FinalMLThickness on "MLT_id" and "Multilayer"
MLScreenMaster = MLScreenMaster.merge(FinalMLThickness, left_on='MLT_id', right_on='MLT_id', how='left')

# Update "cell_tier_group" with the values from "ML Tier"
MLScreenMaster['cell_tier_group'] = MLScreenMaster['ML Tier']

# Drop the extra "Multilayer" and "ML Tier" columns
MLScreenMaster = MLScreenMaster.drop(columns=['Multilayer', 'ML Tier'])

### Plot Multilayer Screen Yield

In [280]:
#Plot Screen Yield
finishedScreen = MLScreenMaster[MLScreenMaster["stage4_finished"] == 1]  # Ensure it's an integer, not a string
# finishedScreen = MLScreenMaster.copy()


samples_to_plot = 'MLB006'
# samples_to_plot = 'FCD00[6,7,8]|FCD011|MLB006|MLD026'
grouping_to_plot = 'WorkWeek'


finishedScreen = finishedScreen[finishedScreen['ML_id'].str.contains(samples_to_plot)]

finishedScreen['TestCycleStart_datetime_first'] = pd.to_datetime(finishedScreen['TestCycleStart_datetime_first'])

finishedScreen['TestCycleStart_datetime_first'] = finishedScreen['TestCycleStart_datetime_first'].dt.strftime('%m/%d/%Y')

########### Filter on Operator Name #############
Operators = pd.read_csv('/Users/mpg01/BaselineOperators.csv')
finishedScreen = finishedScreen.merge(Operators, left_on='ML_id', right_on='samplename', how='left')
finishedScreen.drop(columns=['samplename'], inplace=True)

# finishedScreen =finishedScreen[finishedScreen.WorkWeek.str.contains('W06|W07')]
# finishedScreen = finishedScreen[finishedScreen.OperatorName.str.contains('Gu', na=False)]


grouped = finishedScreen.groupby(grouping_to_plot)






# Calculate normalized bar values for each group
bar_data = []
group_labels = []
for group, df_group in grouped:
    total_rows = len(df_group)
    stage1_count = df_group['stage1_yield'].sum()
    stage2_count = df_group['stage2_yield'].sum()
    stage3_count = df_group['stage3_yield'].sum()
    stage5_count = df_group['stage4_yield'].sum()
    
    # Normalize the values to percentages
    bar_values_percentage = [
        (total_rows / total_rows) * 100,                          # Total rows (always 100%)
        (stage1_count / total_rows) * 100,         # Excluding stage1
        (stage2_count / total_rows) * 100,  # Excluding stage2
        (stage3_count / total_rows) * 100,  # Excluding stage3
        (stage5_count / total_rows) * 100                         # Stage5 (and Stage4) count
    ]
    
    bar_data.append(bar_values_percentage)
    group_labels.append(group)

# Define bar labels
bar_labels = ['Cells Built', 'Initial 1C Yield', 'Fast-Charge Yield', 'Final 1C Yield', 'C/3 Yield']
# Create a new DataFrame for Plotly Express
plot_data = pd.DataFrame(bar_data, columns=bar_labels, index=group_labels)
# Reset index for x-axis labeling
plot_data.reset_index(inplace=True)
plot_data.rename(columns={'index': grouping_to_plot}, inplace=True)

# Define colors for each bar
colors = [
    px.colors.qualitative.Plotly[2],  # Color for 'Total Rows'
    px.colors.qualitative.Plotly[5],  # Color for 'Excluding Stage1'
    px.colors.qualitative.Plotly[3],  # Color for 'Excluding Stage2'
    px.colors.qualitative.Plotly[6],  # Color for 'Excluding Stage3'
    px.colors.qualitative.Plotly[4]   # Color for 'Stage5 and Stage4 Count'
]
# Create the interactive bar chart with Plotly Express, specifying width, height, and custom font sizes
fig = px.bar(
    plot_data, 
    x=grouping_to_plot, 
    y=bar_labels, 
    title="Multilayer Screen Yield", 
    labels={'value': 'Percentage (%)'}, 
    barmode='group', 
    color_discrete_sequence=colors,
    # width=1200,  # Adjust width
    # height=700   # Adjust height
)

# Update layout to increase font sizes
fig.update_layout(
    title=dict(font=dict(size=24)),          # Title font size
    xaxis=dict(title=dict(font=dict(size=18)), tickfont=dict(size=16)),  # X-axis label and ticks
    yaxis=dict(title=dict(font=dict(size=18)), tickfont=dict(size=16)),  # Y-axis label and ticks
    legend=dict(font=dict(size=16)),         # Legend font size
)


# Compute total samples for each group
plot_data["Total Samples"] = plot_data[grouping_to_plot].apply(lambda g: len(finishedScreen[finishedScreen[grouping_to_plot] == g]))

# Convert sample count to text for display
build_count_text = [f"<b>N={n}</b>" for n in plot_data["Total Samples"]]
final_yield_text = [f"<b>{y:.1f}%</b>" for y in plot_data["C/3 Yield"]]

# Assign the text labels to the first and last bars in the grouped bars
fig.data[0].text = build_count_text  # First bar (Cells Built)
fig.data[-1].text = final_yield_text  # Last bar (C/3 Yield)

# Ensure the text is displayed on the bars
fig.update_traces(textposition="inside", textfont_size=18)


# add grey dotted line at 80% yield
fig.add_shape(
    type="line",
    x0=-0.5,
    x1=plot_data.shape[0] - 0.5,
    y0=80,
    y1=80,
    line=dict(color="grey", width=2, dash="dot"),
    )
# Show the plot
fig.show(renderer="browser")

### Plot Cell Metrics

In [281]:
# Modify MLScreenMaster by removing non-yielded cells and cells we had to hardcode
YieldedMLScreenMaster = MLScreenMaster.loc[MLScreenMaster['stage5_yield'] == 1]  # Keep cells that yielded for these plots
YieldedMLScreenMaster = YieldedMLScreenMaster[~YieldedMLScreenMaster['MLT_id'].isin(CellsThatPassedScreen)]  # Filter out the DataFrame to exclude rows where 'MLT_id' is in CellsThatPassedScreen
YieldedMLScreenMaster.reset_index(drop=True, inplace=True)  # Reset the index for the resulting DataFrame (optional)

# Assuming YieldedMLScreenMaster is already loaded and has the necessary columns
color_by = 'Group'  # Color differentiation by Group
grouping = 'Group'

# Add a new numeric column for x positions to shift points
YieldedMLScreenMaster['group_numeric'] = pd.factorize(YieldedMLScreenMaster[grouping])[0]

# Create a color dictionary for each unique value in Group
color = dict(zip(YieldedMLScreenMaster[color_by].unique(), px.colors.qualitative.Plotly * 5))

# Initialize subplot layout
fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.12,
    vertical_spacing=0.1,
    shared_xaxes=True,
    subplot_titles=[
        "Specific Discharge Capacity",
        "Cell Co3 Discharge Capacity"
    ]
)

# Set a flag to ensure legend items are added only once
legend_added = {key: False for key in YieldedMLScreenMaster[color_by].unique()}

# Add traces for the first plot (CellSpecificDischargeCapacity)
for label, group in YieldedMLScreenMaster.groupby(grouping):
    for color_value, group_color in group.groupby(color_by):
        # Add box plot
        fig.add_trace(
            go.Box(
                x=group_color['group_numeric'],  # Use the numeric version of the grouping values
                y=group_color["CellSpecificDischargeCapacity"],
                quartilemethod="linear",
                name=color_value,
                text=group_color["sample_id"] if "sample_id" in group_color.columns else None,
                #showlegend=not legend_added[color_value],
                showlegend=False,
                fillcolor=color[color_value],
                line=dict(color="black"),
            ),
            row=1, col=1
        )

        # Add scatter points for individual data (with a slight shift to the left)
        fig.add_trace(
            go.Scatter(
                x=group_color['group_numeric'] - 0.35,  # Shift numeric positions to the left
                y=group_color["CellSpecificDischargeCapacity"],
                mode="markers",
                marker=dict(color=color[color_value], size=6, symbol='circle'),
                name=f"{color_value} Points",  # Separate legend for points
                legendgroup=color_value,  # Group the legend for boxes and points
                showlegend=False,  # Legend already handled for boxes
            ),
            row=1, col=1
        )
        
        legend_added[color_value] = True

# Reset the legend flags for the second plot
legend_added = {key: False for key in YieldedMLScreenMaster[color_by].unique()}

# Add traces for the second plot (CellCo3DischargeCapcity)
for label, group in YieldedMLScreenMaster.groupby(grouping):
    for color_value, group_color in group.groupby(color_by):
        # Add box plot
        fig.add_trace(
            go.Box(
                x=group_color['group_numeric'],  # Use the numeric version of the grouping values
                y=group_color["CellCo3DischargeCapcity"],
                quartilemethod="linear",
                name=color_value,
                text=group_color["sample_id"] if "sample_id" in group_color.columns else None,
                showlegend=not legend_added[color_value],
                fillcolor=color[color_value],
                line=dict(color="black"),
            ),
            row=1, col=2
        )

        # Add scatter points for individual data (with a slight shift to the left)
        fig.add_trace(
            go.Scatter(
                x=group_color['group_numeric'] - 0.35,  # Shift numeric positions to the left
                y=group_color["CellCo3DischargeCapcity"],
                mode="markers",
                marker=dict(color=color[color_value], size=6, symbol='circle'),
                name=f"{color_value} Points",  # Separate legend for points
                legendgroup=color_value,  # Group the legend for boxes and points
                showlegend=False,  # Legend already handled for boxes
            ),
            row=1, col=2
        )
        
        legend_added[color_value] = True

# Update y-axes titles
fig.update_yaxes(
    title_text="Discharge Capacity (mAh/g)",
    row=1, col=1
)
fig.update_yaxes(
    title_text="Co3 Discharge Capacity (mAh)",
    row=1, col=2
)

# Update x-axes titles
fig.update_xaxes(title_text="Group", row=1, col=1)
fig.update_xaxes(title_text="Group", row=1, col=2)

# Update x-axes ticks to show original grouping labels
fig.update_xaxes(
    tickmode='array',
    tickvals=YieldedMLScreenMaster['group_numeric'].unique(),
    ticktext=YieldedMLScreenMaster[grouping].unique(),
    row=1, col=1,
    ticks="outside",  # Add ticks outside the plot
)
fig.update_xaxes(
    tickmode='array',
    tickvals=YieldedMLScreenMaster['group_numeric'].unique(),
    ticktext=YieldedMLScreenMaster[grouping].unique(),
    row=1, col=2,
    ticks="outside",  # Add ticks outside the plot
)

# Update y-axes ticks to show original grouping labels
fig.update_yaxes(
    ticks="outside",  # Add ticks outside the plot
    row=1, col=1
)
fig.update_yaxes(
    ticks="outside",  # Add ticks outside the plot
    row=1, col=2
)

# Add border to both subplots and adjust layout
fig.update_layout(
    title_text="",
    height=600,
    width=1200,
    showlegend=True,
    xaxis=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for the first subplot
    yaxis=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for the first subplot
    plot_bgcolor='white',  # Set plot background color to white
    paper_bgcolor='white',  # Set paper background color to white
    xaxis2=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for second subplot x-axis (all sides)
    yaxis2=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for second subplot y-axis (all sides)
)

# Display the plot
fig.show(renderer='browser')


In [282]:
# Initialize subplot layout
fig = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.12,
    vertical_spacing=0.1,
    shared_xaxes=True,
    subplot_titles=[
        "dV/dt",
        "Discharge ASR"
    ]
)

# Set a flag to ensure legend items are added only once
legend_added = {key: False for key in YieldedMLScreenMaster[color_by].unique()}

# Add traces for the first plot (Celldvdt)
for label, group in YieldedMLScreenMaster.groupby(grouping):
    for color_value, group_color in group.groupby(color_by):
        # Add box plot
        fig.add_trace(
            go.Box(
                x=group_color['group_numeric'],  # Use the numeric version of the grouping values
                y=group_color["Celldvdt"],
                quartilemethod="linear",
                name=color_value,
                text=group_color["sample_id"] if "sample_id" in group_color.columns else None,
                #showlegend=not legend_added[color_value],
                showlegend=False,
                fillcolor=color[color_value],
                line=dict(color="black"),
            ),
            row=1, col=1
        )

        # Add scatter points for individual data (with a slight shift to the left)
        fig.add_trace(
            go.Scatter(
                x=group_color['group_numeric'] - 0.35,  # Shift numeric positions to the left
                y=group_color["Celldvdt"],
                mode="markers",
                marker=dict(color=color[color_value], size=6, symbol='circle'),
                name=f"{color_value} Points",  # Separate legend for points
                legendgroup=color_value,  # Group the legend for boxes and points
                showlegend=False,  # Legend already handled for boxes
            ),
            row=1, col=1
        )
        
        legend_added[color_value] = True

# Reset the legend flags for the second plot
legend_added = {key: False for key in YieldedMLScreenMaster[color_by].unique()}

# Add traces for the second plot (Cell1CDischargeASR)
for label, group in YieldedMLScreenMaster.groupby(grouping):
    for color_value, group_color in group.groupby(color_by):
        # Add box plot
        fig.add_trace(
            go.Box(
                x=group_color['group_numeric'],  # Use the numeric version of the grouping values
                y=group_color["Cell1CDischargeASR"],
                quartilemethod="linear",
                name=color_value,
                text=group_color["sample_id"] if "sample_id" in group_color.columns else None,
                showlegend=not legend_added[color_value],
                fillcolor=color[color_value],
                line=dict(color="black"),
            ),
            row=1, col=2
        )

        # Add scatter points for individual data (with a slight shift to the left)
        fig.add_trace(
            go.Scatter(
                x=group_color['group_numeric'] - 0.35,  # Shift numeric positions to the left
                y=group_color["Cell1CDischargeASR"],
                mode="markers",
                marker=dict(color=color[color_value], size=6, symbol='circle'),
                name=f"{color_value} Points",  # Separate legend for points
                legendgroup=color_value,  # Group the legend for boxes and points
                showlegend=False,  # Legend already handled for boxes
            ),
            row=1, col=2
        )
        
        legend_added[color_value] = True

# Update y-axes titles
fig.update_yaxes(
    title_text="dV/dt (µV/s)",
    row=1, col=1
)
fig.update_yaxes(
    title_text="1C Discharge ASR (Ohm cm<sup>2</sup>)",
    row=1, col=2
)

# Update x-axes titles
fig.update_xaxes(title_text="Group", row=1, col=1)
fig.update_xaxes(title_text="Group", row=1, col=2)

# Update x-axes ticks to show original grouping labels
fig.update_xaxes(
    tickmode='array',
    tickvals=YieldedMLScreenMaster['group_numeric'].unique(),
    ticktext=YieldedMLScreenMaster[grouping].unique(),
    row=1, col=1,
    ticks="outside",  # Add ticks outside the plot
)
fig.update_xaxes(
    tickmode='array',
    tickvals=YieldedMLScreenMaster['group_numeric'].unique(),
    ticktext=YieldedMLScreenMaster[grouping].unique(),
    row=1, col=2,
    ticks="outside",  # Add ticks outside the plot
)

# Update y-axes ticks to show original grouping labels
fig.update_yaxes(
    ticks="outside",  # Add ticks outside the plot
    row=1, col=1
)
fig.update_yaxes(
    ticks="outside",  # Add ticks outside the plot
    row=1, col=2
)

# Add border to both subplots and adjust layout
fig.update_layout(
    title_text="",
    height=600,
    width=1200,
    showlegend=True,
    xaxis=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for the first subplot
    yaxis=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for the first subplot
    plot_bgcolor='white',  # Set plot background color to white
    paper_bgcolor='white',  # Set paper background color to white
    xaxis2=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for second subplot x-axis (all sides)
    yaxis2=dict(showgrid=True, zeroline=True, showline=True, linecolor='black', linewidth=2, mirror=True, ticks="outside"),  # Border and ticks for second subplot y-axis (all sides)
)

# Display the plot
fig.show(renderer= 'browser')


### Plot Vmin Cycling (Track and 1C)

In [12]:
#Plot Vmin during track cycling
# Order dataframe so it appears nicely in Vmin Reliability Plot
MLTrackCycleEach = MLTrackCycleEach.sort_values(by=['sample_id', 'ReliabilityCycles'])

# Create a scatter/line plot
fig = px.scatter(
    MLTrackCycleEach,
    x='ReliabilityCycles',
    y='min_track_cycle_voltage',
    color='Group',
    title="Minimum Cycle Voltage in Track Cycle Reliability",
    labels={'ReliabilityCycles': 'Cycle Count', 'min_track_cycle_voltage': 'Voltage (V)'},
    hover_data={"sample_id": True} 
)

# Update traces to include smaller points and lines
fig.update_traces(mode='markers+lines', marker=dict(size=8), line=dict(width=2))

# Customize axes
fig.update_yaxes(
    #range=[2.2, 3.1],
    title='Voltage (V)',
    tickfont=dict(size=22),
    titlefont=dict(size=22),
    mirror=True,
    ticks='outside',
    showline=True,
    linewidth=2,
    linecolor='grey'
)
fig.update_xaxes(
    range=[0, 100],
    title='Cycle Number',
    tickfont=dict(size=22),
    titlefont=dict(size=22),
    mirror=True,
    ticks='outside',
    showline=True,
    linewidth=2,
    linecolor='grey'
)

# Add a horizontal dotted line at 2.45V
fig.add_shape(
    type="line",
    x0=0,
    y0=2.45,
    x1=100,
    y1=2.45,
    line=dict(color="black", width=0.5, dash="dot")
)

# Update layout for a clean appearance
fig.update_layout(
    autosize=False,
    width=900,
    height=600,
    font=dict(size=20),
    plot_bgcolor='white'
)

# Show the plot
fig.show(renderer='browser')


In [122]:
#Plot Vmin during 1C cycling
# Order dataframe so it appears nicely in Vmin Reliability Plot
MLslctCycleEach = MLslctCycleEach.sort_values(by=['MLT_id', 'ReliabilityCycles'])

samples_to_plot = 'MLD|FCD'

MLslctCycleEachFiltered = MLslctCycleEach[MLslctCycleEach['MLT_id'].str.contains(samples_to_plot)]

MLslctCycleEachFiltered = MLslctCycleEachFiltered[MLslctCycleEachFiltered.CycleFailure == False]

MLslctCycleEachFiltered = MLslctCycleEachFiltered[MLslctCycleEachFiltered.DischargeCapacityRetention >0.80]

# Create a scatter/line plot
fig = px.scatter(
    MLslctCycleEachFiltered,
    x='ReliabilityCycles',
    y='DischargeCapacityRetention',
    color="MLT_id",
    title="Discharge Capacity Retention of 1C Cycle Reliability",
    labels={'ReliabilityCycles': 'Cycle Count', 'DischargeCapacityRetention': 'Capacity Retention'},
    hover_data={"MLT_id": True} 
)

# Update traces to include smaller points and lines
fig.update_traces(mode='markers+lines', marker=dict(size=8), line=dict(width=2))

# Customize axes
fig.update_yaxes(
    range=[0.8, 1.01],
    title='Discharge Capacity Retention',
    tickfont=dict(size=22),
    titlefont=dict(size=22),
    mirror=True,
    ticks='outside',
    showline=True,
    linewidth=2,
    linecolor='grey'
)
fig.update_xaxes(
    range=[0, 100],
    title='Cycle Number',
    tickfont=dict(size=22),
    titlefont=dict(size=22),
    mirror=True,
    ticks='outside',
    showline=True,
    linewidth=2,
    linecolor='grey'
)

# Add a horizontal dotted line at 2.45V
fig.add_shape(
    type="line",
    x0=0,
    y0=2.45,
    x1=100,
    y1=2.45,
    line=dict(color="black", width=0.5, dash="dot")
)

# Update layout for a clean appearance
fig.update_layout(
    autosize=False,
    width=1000,
    height=600,
    font=dict(size=20),
    plot_bgcolor='white'
)

# Show the plot
fig.show(renderer='browser')


### Plot Reliability Survival

In [288]:
#Plot Track Cycle Survival Curve
# Sort the DataFrame to prioritize rows based on the desired order
MLTrackCycleEach = MLTrackCycleEach.sort_values(
    by=['sample_id', 'track_cycle_count_cumulative', 'TestCycleStart_datetime'], 
    ascending=[True, False, False]
)

grouping_to_plot = 'Group'


samples_to_plot = "FCD008"


samples_to_exclude_plot = "MLD026AA-PS00-05"

MLTrackCycleEachFiltered = MLTrackCycleEach[MLTrackCycleEach.sample_id.str.contains(samples_to_plot)]


if samples_to_exclude_plot:
    MLTrackCycleEachFiltered = MLTrackCycleEachFiltered [~MLTrackCycleEach.sample_id.str.contains(samples_to_exclude_plot)]
    

# Merge df_master with FinalMLTier on "samplename" and "Multilayer"
MLTrackCycleEachFiltered= MLTrackCycleEachFiltered.merge(FinalMLTier, left_on='sample_id', right_on='Multilayer', how='left')
MLTrackCycleEachFiltered.drop(columns=['Multilayer'], inplace=True)

# Merge df_master with FinalMLThickness on "MLT_id" and "Multilayer"
MLTrackCycleEachFiltered = MLTrackCycleEachFiltered.merge(FinalMLThickness, left_on='sample_id', right_on='MLT_id', how='left')
MLTrackCycleEachFiltered.drop(columns=['MLT_id'], inplace=True)

MLTrackCycleEachFiltered.loc[MLTrackCycleEachFiltered.sample_id.str.contains('MLD018AB-PS00-01|MLD018AC-PS00-03|MLD018AD-PS00-03|MLD018AE-PS00-01|MLD018AF-PS00-01'), 'Condition'] = 'Standard TC'
MLTrackCycleEachFiltered.loc[MLTrackCycleEachFiltered.sample_id.str.contains('MLD016AA-PS00-03|MLD016AB-PS00-03|MLD018AD-PS00-02|MLD018AB-PS00-02|MLD018AE-PS00-02|MLD018AF-PS00-03'), 'Condition'] = 'FD TC'

MLTrackCycleEachFiltered.loc[MLTrackCycleEachFiltered.sample_id.str.contains('MLD007A[U,V,W]|MLD025|FCD008'), 'Condition'] = 'Jetting POR'
MLTrackCycleEachFiltered.loc[MLTrackCycleEachFiltered.sample_id.str.contains('FCD006A[B,C]'), 'Condition'] = 'Jetting (No Terminal Layer)'
MLTrackCycleEachFiltered.loc[MLTrackCycleEachFiltered.sample_id.str.contains('MLB006'), 'Condition'] = 'Baseline'

### Filter rows by quality metrics

# MLTrackCycleEachFiltered=MLTrackCycleEachFiltered[MLTrackCycleEachFiltered['center_normalized_0_5mm_eroded_rect_outside_median_us']<1.04]


def select_cycle_count(group):
    # Check if any of the failure conditions are met and track_cycle_count_cumulative > 0
    failure_conditions = (group[['dvdt_failure', 'min_track_cycle_voltage_failure', 'CapacityChargeFraction_failure']] == 1).any(axis=1)
    failure_rows = group[failure_conditions & (group["track_cycle_count_cumulative"] > 0)]
    
    if not failure_rows.empty:
        selected_row = failure_rows.nsmallest(1, "track_cycle_count_cumulative")  # Keep the first lowest non-zero failure
        selected_row["Shorted"] = True  # Mark as shorted
    else:
        selected_row = group.nlargest(1, "track_cycle_count_cumulative")  # Keep the highest if no failure is found
        selected_row["Shorted"] = False  # Mark as not shorted
    
    return selected_row

MLLastTrackCycle = MLTrackCycleEachFiltered.groupby('sample_id', group_keys=False).apply(select_cycle_count)

# Ensure TestCycleEnd_date is in datetime format
MLLastTrackCycle['TestCycleEnd_date'] = pd.to_datetime(MLLastTrackCycle['TestCycleEnd_date'])

# delete rows that have Nans for track_cycle_count_cumulative
MLLastTrackCycle = MLLastTrackCycle.dropna(subset=['track_cycle_count_cumulative'])

# Get today's date in the same format
today_date = pd.to_datetime(datetime.today().date())

# Update 'Shorted' to False if TestCycleEnd_date is today
#MLLastTrackCycle.loc[MLLastTrackCycle['TestCycleEnd_date'] == today_date, 'Shorted'] = False




# # Create an expanded DataFrame to allow for overlapping group assignments
# def expand_groups(row):
#     groups = []
#     if row["center_normalized_0_5mm_eroded_rect_outside_median_us"] < 1.04:
#         groups.append("<1.04")
#     if row["center_normalized_0_5mm_eroded_rect_outside_median_us"] < 1.05:
#         groups.append("<1.05")  # Includes <1.04
#     if row["center_normalized_0_5mm_eroded_rect_outside_median_us"] > 1.05:
#         groups.append(">1.05")

#     return groups

# # Explode the DataFrame to allow multiple group assignments per sample
# MLLastTrackCycle["Group"] = MLLastTrackCycle.apply(expand_groups, axis=1)
# MLLastTrackCycle = MLLastTrackCycle.explode("Group")  # Duplicates rows for overlapping groups

# grouping_to_plot = 'Group'







plotly_colors = [
    # 'rgb(99, 110, 250)',    # Blue
    'rgb(239, 85, 59)',     # Red-orange
    'rgb(0, 204, 150)',     # Green
    'rgb(171, 99, 250)',    # Purple
    'rgb(255, 161, 90)',    # Orange
    'rgb(25, 211, 243)',    # Cyan
    'rgb(255, 102, 146)',   # Pink
    'rgb(182, 232, 128)',   # Light green
    'rgb(255, 151, 255)',   # Light pink
    'rgb(254, 203, 82)'     # Yellow-orange
]*10

# Create figure
fig = make_subplots()
color_list = plotly_colors
fill_color_list = ['rgba' + c[3:-1] + ', 0.15)' for c in color_list]
i=0

RMST_duration=60
kmf1 = KaplanMeierFitter(alpha=0.05)  # this alpha is the Type I error rate
results_df = pd.DataFrame(columns=['Condition', 'RMST', 'Variance', '95_CI'])
range_x = 60



# results_df = pd.DataFrame(columns=['Condition', 'RMST', 'Variance', '95_CI'])

# df_rel_master_voltage_summary=pd.read_csv('Reliability_summary.csv')

# MLLastTrackCycle = pd.read_csv('MLLastTrackCycle.csv')

for Batch, grouped in MLLastTrackCycle.groupby(grouping_to_plot):
    
    kmf1.fit(durations=grouped["track_cycle_count_cumulative"], event_observed=grouped["Shorted"])
    df = kmf1.survival_function_.join(kmf1.confidence_interval_survival_function_)
    df = df.join(
        grouped.set_index("sample_id")[["track_cycle_count_cumulative", "Shorted"]]
        .reset_index()
        .groupby(["track_cycle_count_cumulative", "Shorted"])
        .agg({"sample_id": "<br>\n".join})
        .reset_index()
        .set_index("track_cycle_count_cumulative")
    )



    df = df.fillna(value=True)
    df["Shorted"] = df["Shorted"].apply(int)
    df.loc[df.index==0, "Shorted" ] = 0
    df['color']=color_list[i]


    # Calculate RMST and variance
    rmst, variance = restricted_mean_survival_time(kmf1, t=RMST_duration, return_variance=True)
    
    standard_error = np.sqrt(variance)
    z_score = 1.96  # for 95% confidence interval

    # Compute confidence intervals
    ci = z_score * standard_error
    

    # Append results to the DataFrame
    results_df = results_df.append({'Condition': Batch,
                                    'RMST': rmst,
                                    'Variance': variance,
                                    '95_CI': ci}, ignore_index=True)
    

    # if six_layer:
    #     if '6L' not in Batch:
    #         df['KM_estimate']=df['KM_estimate']**3
    #         df['KM_estimate_lower_0.95']=df['KM_estimate_lower_0.95']**3
    #         df['KM_estimate_upper_0.95']=df['KM_estimate_upper_0.95']**3


    trace1 = {
        "x": df.index,
        "y": df.KM_estimate,
        "line": {"shape": "hv"},
        "mode": "lines",
        "name": "value",
        "type": "scatter",
    }

    df=df.dropna(subset='Shorted')



    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df.KM_estimate * 100,
            mode="markers+lines",
            line=dict(shape="hv", width=3, color=color_list[i]),
            marker=dict(color=df['color'], symbol='circle', size=7*(1-df['Shorted'])),
            hovertext=df.sample_id,
            name=f"{Batch} (N={len(grouped)})",
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["KM_estimate_upper_0.95"] * 100,
            mode="lines",
            line=dict(shape="hv", width=0, color=color_list[i]),
            name="",  # f"{Batch} UCI95%",
            showlegend=False,
        ),
        secondary_y=False,
    )
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df["KM_estimate_lower_0.95"] * 100,
            mode="lines",
            fill="tonexty",
            fillcolor=fill_color_list[i],
            line=dict(shape="hv", width=0, color=color_list[i]),
            name="",  # f"{Batch} LCI95%",
            showlegend=False,
        ),
        secondary_y=False,
    )
    i+=1



fig.update_layout(
    title="6L Reliability",
    xaxis=dict(title="Cycle Number"),
    yaxis=dict(title="Survival (%)"),
    font=dict(size=20),
    legend={"traceorder": "normal"},
    legend_title_text=grouping_to_plot,
    # autosize=False,S
    width=1275,
    height=700,
    # hide the legend
    # showlegend=False,
)

# set background color to white
fig.update_layout(plot_bgcolor='white')
fig.update_yaxes(range=[0, 105], showline=True, linewidth=1, linecolor="black", mirror=True)
fig.update_xaxes(range=[0, 60], showline=True, linewidth=1, linecolor="black", mirror=True)



# add vertical grey dashed line to figure at 60 cycles
fig.add_shape(
        # Line Vertical
        dict(
            type="line",
            x0=60,
            y0=0,
            x1=60,
            y1=105,
            line=dict(
                color="Grey",
                width=3,
                dash="dash",
            ),
        )
    )

fig.add_trace(
go.Scatter(
    x=[10],
    y=[95],
    mode="markers",
    marker=dict(color="red", symbol="circle", size=20),
    hovertext='95% Survival',
    name="",
    # remove from legend
    showlegend=False,
),
secondary_y=False,
)
# adding gate 2 22L target
fig.add_trace(
go.Scatter(
    x=[60],
    y=[95],
    mode="markers",
    marker=dict(color="blue", symbol="circle", size=20),
    hovertext='95% Survival',
    name="",
    # remove from legend
    showlegend=False,
),
secondary_y=False,
)



fig.update_yaxes(range=[0, 105])
fig.update_xaxes(range=[0, range_x])

fig.show(renderer="browser")




# # Initialize Kaplan-Meier fitter
# kmf = KaplanMeierFitter(alpha=0.05)

# # Offset for unique x-values to prevent overlapping indices
# x_offset = 1e-6
# #x_offset = 1

# # Iterate over groups in MLLastTrackCycle
# for i, (group, group_data) in enumerate(MLLastTrackCycle.groupby('Group')):
#     # Prepare data for Kaplan-Meier fitting
#     durations = group_data['track_cycle_count_cumulative'].to_numpy()
#     event_observed = group_data['Shorted'].to_numpy()

#     # Fit Kaplan-Meier model  
#     kmf.fit(durations=durations, event_observed=event_observed)

#     # Add survival curve trace
#     fig.add_trace(
#         go.Scatter(
#             x=kmf.survival_function_.index + (i * x_offset),  # Add offset to x-values
#             y=kmf.survival_function_['KM_estimate'] * 100,
#             mode="lines",
#             line=dict(shape="hv", width=3, color=color_list[i]),
#             name=f"{group} (N={len(group_data)})",
#         )
#     )

#     # Add confidence interval trace
#     ci = kmf.confidence_interval_survival_function_
    

#     if not ci.empty:
#         fig.add_trace(
#             go.Scatter(
#                 x=np.concatenate([ci.index, ci.index[::-1]]) + (i * x_offset),  # Offset x-values
#                 y=np.concatenate([
#                     ci['KM_estimate_upper_0.95'] * 100,
#                     (ci['KM_estimate_lower_0.95'] * 100)[::-1]
#                 ]),
#                 mode="lines",
#                 fill="toself",
#                 fillcolor=fill_color_list[i],
#                 line=dict(color='rgba(255,255,255,0)'),
#                 showlegend=False,
#             )
#         )

#     # Add markers for hover labels
#     predicted = kmf.predict(durations)
#     survival_probabilities = np.array(predicted) if isinstance(predicted, (np.ndarray, pd.Series)) else np.array([predicted])

#     customdata = np.stack(
#         [
#             group_data['sample_id'].to_numpy(),  # Sample ID
#             group_data['ReliabilityCycles'].to_numpy(),  # Reliability cycles
#             survival_probabilities * 100  # Survival percentage
#         ],
#         axis=-1
#     )

#     # Add marker trace for hover
#     fig.add_trace(
#     go.Scatter(
#         x=durations + (i * x_offset),  # Add offset to x-values
#         y=survival_probabilities * 100,
#         mode="markers",
#         marker=dict(size=8, symbol="circle", color=color_list[i]),
#         customdata=customdata,
#         hovertemplate=(
#             "<b>Sample ID:</b> %{customdata[0]}<br>"
#             "<b>Reliability Cycles:</b> %{customdata[1]}<br>"
#             "<b>Survival %:</b> %{customdata[2]:.2f}%"
#         ),
#         hoverinfo="x+y+name+text",  # Specify exactly what to show in the hover
#         name=f"{group} Markers",  # Unique name for each group
#         showlegend=False,  # Prevent marker legends from cluttering
#     )
# )




# # Finalize layout
# fig.update_layout(
#     title="Track Cycle Survival Plot by Group",
#     xaxis_title="Cycle Count",
#     yaxis_title="Survival Probability (%)",
#     legend_title="Groups",
#     font=dict(size=14),
#     plot_bgcolor="white",
#     width=900,
#     height=600
# )

# fig.add_shape(
#             # Line Vertical
#             dict(
#                 type="line",
#                 x0=60,
#                 y0=0,
#                 x1=60,
#                 y1=105,
#                 line=dict(
#                     color="Grey",
#                     width=3,
#                     dash="dash",
#                 ),
#             )
#         )
# fig.add_trace(
#     go.Scatter(
#         x=[10],
#         y=[95],
#         mode="markers",
#         marker=dict(color="red", symbol="circle", size=20),
#         hovertext='95% Survival',
#         name="",
#         # remove from legend
#         showlegend=False,
#     ),
#     secondary_y=False,
#     )
#     # adding gate 2 22L target
# fig.add_trace(
#     go.Scatter(
#         x=[60],
#         y=[95],
#         mode="markers",
#         marker=dict(color="blue", symbol="circle", size=20),
#         hovertext='95% Survival',
#         name="",
#         # remove from legend
#         showlegend=False,
#     ),
#     secondary_y=False,
#     )

# fig.update_yaxes(range=[0, 105], showline=True, linewidth=1, linecolor="black", mirror=True)
# fig.update_xaxes(range=[0, 150], showline=True, linewidth=1, linecolor="black", mirror=True)

# fig.show()



/var/folders/w3/vd6ll12160n19dfsmnsbc1rnhgdmtf/T/ipykernel_20485/1625288747.py:20: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [222]:
MLLastTrackCycle.to_csv('MLLastTrackCycle.csv', index=False)

In [175]:
MLLastTrackCycle = pd.read_csv('MLLastTrackCycle.csv')

In [120]:
#Plot 1C Cycle Survival Curve
# Sort the DataFrame to prioritize rows based on the desired order
MLslctCycleEach = MLslctCycleEach.sort_values(
    by=['MLT_id', 'ReliabilityCycles', 'TestCycleStart_datetime'], 
    ascending=[True, False, False]
)


MLslctCycleSummary.loc[MLslctCycleSummary.MLT_id.str.contains('MLD|FCD'), 'Condition'] = '22L 1C Cycling'

grouping_to_plot = 'Condition'

def select_cycle_count(group):
    # Check if any of the failure conditions are met and track_cycle_count_cumulative > 0
    failure_conditions = (group[['dvdt_failure', 'CapacityChargeFraction_failure']] == 1).any(axis=1)
    failure_rows = group[failure_conditions & (group["ReliabilityCycles"] > 0)]
    
    if not failure_rows.empty:
        selected_row = failure_rows.nsmallest(1, "ReliabilityCycles")  # Keep the first lowest non-zero failure
        selected_row["Shorted"] = True  # Mark as shorted
    else:
        selected_row = group.nlargest(1, "ReliabilityCycles")  # Keep the highest if no failure is found
        selected_row["Shorted"] = False  # Mark as not shorted
    
    return selected_row

MLLastslctCycle = MLslctCycleEach.groupby('MLT_id', group_keys=False).apply(select_cycle_count)

# Ensure TestCycleEnd_date is in datetime format
MLLastslctCycle['TestCycleEnd_date'] = pd.to_datetime(MLLastslctCycle['TestCycleEnd_date'])



##Determine which cells in MLTrackCycleSummary shorted
MLslctCycleSummary['Shorted'] = MLslctCycleSummary['CycleFailure']


# Get today's date in the same format
today_date = pd.to_datetime(datetime.today().date())

# Update 'Shorted' to False if TestCycleEnd_date is today
#MLLastTrackCycle.loc[MLLastTrackCycle['TestCycleEnd_date'] == today_date, 'Shorted'] = False


plotly_colors = [
    # 'rgb(99, 110, 250)',    # Blue
    'rgb(239, 85, 59)',     # Red-orange
    'rgb(0, 204, 150)',     # Green
    'rgb(171, 99, 250)',    # Purple
    'rgb(255, 161, 90)',    # Orange
    'rgb(25, 211, 243)',    # Cyan
    'rgb(255, 102, 146)',   # Pink
    'rgb(182, 232, 128)',   # Light green
    'rgb(255, 151, 255)',   # Light pink
    'rgb(254, 203, 82)'     # Yellow-orange
]*10

# Create figure
fig = make_subplots()
color_list = plotly_colors
fill_color_list = ['rgba' + c[3:-1] + ', 0.15)' for c in color_list]

# Initialize Kaplan-Meier fitter
kmf = KaplanMeierFitter(alpha=0.05)

# Offset for unique x-values to prevent overlapping indices
x_offset = 1e-6
#x_offset = 1

# Iterate over groups in MLslctCycleSummary
for i, (group, group_data) in enumerate(MLslctCycleSummary.groupby(grouping_to_plot)):
    # Prepare data for Kaplan-Meier fitting
    durations = group_data['ReliabilityCycles'].to_numpy()
    event_observed = group_data['Shorted'].to_numpy()

    # Fit Kaplan-Meier model  
    kmf.fit(durations=durations, event_observed=event_observed)

    # Add survival curve trace
    fig.add_trace(
        go.Scatter(
            x=kmf.survival_function_.index + (i * x_offset),  # Add offset to x-values
            y=kmf.survival_function_['KM_estimate'] * 100,
            mode="lines",
            line=dict(shape="hv", width=3, color=color_list[i]),
            name=f"{group} (N={len(group_data)})",
        )
    )

    # Add confidence interval trace
    ci = kmf.confidence_interval_survival_function_
      
    if not ci.empty:
        fig.add_trace(
            go.Scatter(
                x=np.concatenate([ci.index, ci.index[::-1]]) + (i * x_offset),
                y=np.concatenate([
                    ci['KM_estimate_upper_0.95'] * 100,
                    (ci['KM_estimate_lower_0.95'] * 100)[::-1]
                ]),
                mode="lines",
                fill="toself",
                fillcolor=fill_color_list[i],
                line=dict(color='rgba(255,255,255,0)'),
                showlegend=False,
                hoverinfo="skip"  # <- Prevents hover over CI
            )
        )

    # Add markers for hover labels
    predicted = kmf.predict(durations)
    survival_probabilities = np.array(predicted) if isinstance(predicted, (np.ndarray, pd.Series)) else np.array([predicted])

    customdata = np.stack(
        [
            group_data['MLT_id'].to_numpy(),  # Sample ID
            group_data['ReliabilityCycles'].to_numpy(),  # Reliability cycles
            survival_probabilities * 100  # Survival percentage
        ],
        axis=-1
    )

    # Add marker trace for hover
    fig.add_trace(
    go.Scatter(
        x=durations + (i * x_offset),  # Add offset to x-values
        y=survival_probabilities * 100,
        mode="markers",
        marker=dict(size=8, symbol="circle", color=color_list[i]),
        customdata=customdata,
        hovertemplate=(
            "<b>Sample ID:</b> %{customdata[0]}<br>"
            "<b>Reliability Cycles:</b> %{customdata[1]}<br>"
            "<b>Survival %:</b> %{customdata[2]:.2f}%"
        ),
        hoverinfo="x+y+name+text",  # Specify exactly what to show in the hover
        name=f"{group} Markers",  # Unique name for each group
        showlegend=False,  # Prevent marker legends from cluttering
    )
)




# Finalize layout
fig.update_layout(
    title="1C Cycle Survival Plot by Group",
    xaxis_title="Cycle Count",
    yaxis_title="Survival Probability (%)",
    legend_title="Groups",
    font=dict(size=14),
    plot_bgcolor="white",
    width=900,
    height=600
)

fig.add_shape(
            # Line Vertical
            dict(
                type="line",
                x0=60,
                y0=0,
                x1=60,
                y1=105,
                line=dict(
                    color="Grey",
                    width=3,
                    dash="dash",
                ),
            )
        )
#fig.add_trace(
#    go.Scatter(
#        x=[10],
#        y=[95],
#        mode="markers",
#        hovertext='95% Survival',
#        name="",
#        # remove from legend
#        showlegend=False,
#    ),
#    secondary_y=False,
#    )
    # adding gate 2 22L target
#fig.add_trace(
#    go.Scatter(
#        x=[60],
#        y=[95],
#        mode="markers",
#        marker=dict(color="blue", symbol="circle", size=20),
#        hovertext='95% Survival',
#        name="",
#        # remove from legend
#        showlegend=False,
#    ),
#    secondary_y=False,
#    )

fig.update_yaxes(range=[0, 105], showline=True, linewidth=1, linecolor="black", mirror=True)
fig.update_xaxes(range=[0, 100], showline=True, linewidth=1, linecolor="black", mirror=True)

fig.show(renderer='browser')


### Produce ML Tracker Spreadsheet

In [19]:
# Rename columns and assign to df_screening
df_screening = MLScreenMaster[
    ['MLT_id', 'cell_tier_group', 'CycleFailure', 'TestCycleStart_date', 'ElectricalTestTool', 'ElectricalTestChannel']
].rename(
    columns={
        'MLT_id': 'MultilayerID',
        'cell_tier_group': 'Tier',
        'CycleFailure': 'ML Screen Yield',
        'TestCycleStart_date': 'Build Date',
        'ElectricalTestTool': 'Tool',
        'ElectricalTestChannel': 'Channel'
    }
)

# Relabel 'false' to 'Pass' and 'true' to 'Failed'
df_screening['ML Screen Yield'] = df_screening['ML Screen Yield'].replace({False: 'Pass', True: 'Fail'})


# Merge 'ML___CycleSummary' into 'df_screening'
df_screening = pd.merge(
    df_screening,
    MLTrackCycleSummary[['sample_id', 'ElectricalTestTool', 'ElectricalTestChannel', 'CycleFailure', 'ReliabilityCycles', 'Test_Version']],
    left_on='MultilayerID',  # Column in 'df_screening'
    right_on='sample_id',    # Column in 'MLLastTrackCycle'
    how='left'               # Use 'left' join to keep all rows from 'df_screening'
)
# Overwrite 'Tool' and 'Channel' where 'ElectricalTestTool' and 'ElectricalTestChannel' have values
df_screening['Tool'] = df_screening['ElectricalTestTool'].combine_first(df_screening['Tool'])
df_screening['Channel'] = df_screening['ElectricalTestChannel'].combine_first(df_screening['Channel'])
# Drop the additional columns no longer needed
df_screening = df_screening.drop(columns=['ElectricalTestTool', 'ElectricalTestChannel', 'sample_id'])

# Merge df_screening with MLslctCycleSummary based on matching MultilayerID and sample_id
if not MLslctCycleSummary.empty:
    df_screening.update(
        df_screening.set_index('MultilayerID')
        .combine_first(MLslctCycleSummary.set_index('ML_id'))
        .reset_index()
    )




# Rename 'CycleFailure' to 'Cycle Status'
df_screening = df_screening.rename(columns={'CycleFailure': 'Cycle Status', 'Test_Version':'Cycling Test Version'})

# Replace 'True' with 'Shorted' and 'False' with 'Live' in the 'Cycle Status' column
df_screening['Cycle Status'] = df_screening['Cycle Status'].replace({True: 'Shorted', False: 'Live'})

#save the updated dataframe
Output_name = 'ML_TrackingResults.xlsx'
df_screening.to_excel(Output_name, index=False)
